<a id="segmentation-and-sampling"></a>
# VideoDB Understanding: Segmentation and Sampling

Control how a video becomes time ranges and which frames each visual analyzer receives.


<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/segmentation-and-sampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install, connect, and choose a video

In [ ]:
!pip install -q videodb python-dotenv pandas

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
collection = conn.get_collection()
print("Connected to VideoDB")
print("Collection:", collection.id)

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

video = collection.upload(VIDEO_URL)

# To use an existing video instead:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Video:", video.id)
video.play()

<a id="segmentation"></a>
## 2. Segmentation

- **Shot:** boundaries follow visual cuts. Tune with `threshold` and optionally `min_scene_len`.
- **Time:** fixed windows controlled by `seconds`.

Segmentation is shared by the analyzers in one Understanding run.

In [ ]:
SHOT_SEGMENTATION = {"type": "shot", "threshold": 30, "min_scene_len": 15}
TIME_SEGMENTATION = {"type": "time", "seconds": 20}

print("Segmentation strategies:")
print(f"- Shot: {SHOT_SEGMENTATION}")
print(f"- Time: {TIME_SEGMENTATION}")

<a id="sampling"></a>
## 3. Sampling

- **Interval:** take a frame every N seconds; useful for dense monitoring.
- **Uniform:** take a fixed number of frames from each segment; useful for scene descriptions.

Sampling can be global or overridden on a visual analyzer. Speech transcription does not use frame sampling.

In [ ]:
INTERVAL_SAMPLING = {"strategy": "interval", "every": 2}
UNIFORM_SAMPLING = {"strategy": "uniform", "frame_count": 3}

print("Sampling strategies:")
print(f"- Interval: {INTERVAL_SAMPLING}")
print(f"- Uniform: {UNIFORM_SAMPLING}")

## 4. Compare shot and time segmentation

The same activity analyzer is run twice so only the segmentation changes.

In [ ]:
def run_activity(segmentation):
    return video.understand(
        analyzers=[{
            "type": "activity_recognition",
            "name": "activity",
            "sampling": {"strategy": "uniform", "frame_count": 3},
            "config": {"model": "ultra"},
        }],
        segmentation=segmentation,
    )


shot_run = run_activity(SHOT_SEGMENTATION)
time_run = run_activity(TIME_SEGMENTATION)

for run in (shot_run, time_run):
    run.wait_until_complete(timeout=3600, poll_interval=15)

print("Run statuses:")
print(f"- Shot segmentation: {shot_run.status}")
print(f"- Time segmentation: {time_run.status}")

In [ ]:
import pandas as pd


def activity_rows(run, strategy):
    output = run.get_analyzer("activity").get_output()
    return [
        {
            "strategy": strategy,
            "start": scene.get("start"),
            "end": scene.get("end"),
            "activity": (scene.get("data") or {}).get("activity"),
        }
        for scene in output.get("scenes", [])
    ]


comparison = pd.DataFrame(
    activity_rows(shot_run, "shot") + activity_rows(time_run, "time")
)
print(f"Comparison rows: {len(comparison)}")
comparison

## 5. Selection guide

| Goal | Segmentation | Sampling |
|---|---|---|
| Scene descriptions | Shot | Uniform, 3–8 frames |
| Periodic monitoring | Time | Interval |
| OCR/brand checks | Shot or time | Uniform for overview; interval for recall |
| Object tracking-like coverage | Time | Short interval |
| Long-video summaries | Longer time windows | Uniform |

More frames improve coverage but increase latency and model usage. Start sparse, inspect misses, and increase sampling deliberately.

## Optional cleanup

In [ ]:
DELETE_RUNS = False
if DELETE_RUNS:
    shot_run.delete()
    time_run.delete()